# Tutorial 6 — Mini-RAG: perguntas e fontes documentais

**Nível:** Intermediário · **Tempo:** 50 min

**Pergunta:** Como recuperar evidências em uma coleção de documentos populacionais antes de elaborar respostas?

> Dados didáticos sintéticos, não constituem evidência empírica.

## Passo 1 — Criar documentos fictícios

São definições simplificadas e fictícias; não são citações de fontes científicas.

In [ ]:
documentos={"DOC-01":"A população idosa é definida neste exemplo como pessoas de 60 anos ou mais. O indicador é calculado dividindo esse grupo pelo total.","DOC-02":"O saldo migratório corresponde à diferença entre entradas e saídas de pessoas em determinado território e período.","DOC-03":"A taxa bruta de natalidade relaciona o número de nascidos vivos à população média, geralmente por mil habitantes.","DOC-04":"Projeções populacionais dependem de hipóteses sobre fecundidade, mortalidade e migração."}
print("Documentos:",list(documentos))

## Passo 2 — Representar e indexar textos

TF-IDF é uma técnica lexical de recuperação. Não equivale a embeddings semânticos de LLM.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
ids=list(documentos)
textos=[documentos[i] for i in ids]
vec=TfidfVectorizer(strip_accents="unicode",ngram_range=(1,2))
matriz=vec.fit_transform(textos)
print("Matriz de documentos:",matriz.shape)

## Passo 3 — Recuperar trechos com evidências

A pontuação serve à ordenação dos trechos, não como probabilidade de a resposta estar correta.

In [ ]:
pergunta="Quais fatores entram nas projeções populacionais?"
q=vec.transform([pergunta])
pontuacoes=cosine_similarity(q,matriz)[0]
ranking=sorted(zip(ids,pontuacoes),key=lambda p:-p[1])
for id,score in ranking[:3]:print(id,round(float(score),3),documentos[id])

## Passo 4 — Construir contexto rastreável

A prompt de síntese inclui somente trechos selecionados com identificadores.

In [ ]:
melhores=[(i,s) for i,s in ranking[:2] if s>0]
contexto="\n".join(f"[{i}] {documentos[i]}" for i,_ in melhores)
prompt=f"""Responda usando EXCLUSIVAMENTE os trechos.
Cite o identificador do documento para cada afirmação.
Se não houver resposta, escreva: Não consta nos documentos.
PERGUNTA: {pergunta}
FONTES:
{contexto}"""
print(prompt)

## Passo 5 — Construir resposta verificável sem LLM

A resposta extrativa funciona sem API, mas pode não atender perguntas complexas.

In [ ]:
if melhores:
    print("Evidência mais próxima:",melhores[0][0])
    print(documentos[melhores[0][0]])
else:
    print("Não foram encontrados trechos correspondentes.")

## Passo 6 — Gerar resposta com LLM opcional

Um LLM pode omitir ou inventar referências; confira toda resposta contra os trechos recuperados.

In [ ]:
USAR_LLM=False
if USAR_LLM:
    from transformers import pipeline
    gerador=pipeline("text2text-generation",model="google/flan-t5-small",device=-1)
    print(gerador(prompt,max_new_tokens=120)[0]["generated_text"])
else:
    print("Geração opcional desativada; recuperação documental demonstrada.")

## Agora é sua vez

Troque a pergunta por “O que é saldo migratório?” e “Qual foi a população do Brasil em 2022?”. No segundo caso, a resposta deve reconhecer a ausência de evidência.

## Limitações e cuidados

Coleção artificial pequena; recuperação lexical depende do vocabulário. RAG melhora a fundamentação potencial, mas não garante fidelidade, precisão ou completude.